# Module 6: Subagent Handoffs & Deterministic Behavior

> Part of the **Modular Workshops** series. Standalone, ~35 min.

Two recurring design questions from teams putting agents in production, answered with the actual tradeoffs (and made concrete in traces) rather than a preference:

**Part 1 — Subagent handoffs.** *"What's the sweet spot for handing a task to a subagent — how much context do you share so it's not bloated or overloaded?"* There isn't one answer; there are three patterns with different costs. We build all three on the shopping assistant, then open a trace and read per-subagent tokens/latency/context so you can *see* whether a subagent is earning its tokens or a hop is losing context.

**Part 2 — Deterministic behavior.** *"Can I make the agent more deterministic for specific pathways, so a repeat query does fewer tool calls and less re-deciding?"* Determinism is a **dial**, not a feature. We go down it in order: put determinism in code where it matters → constrain the model where it stays agentic → the self-improving loop (Engine) → version the behavior so a change is promotable and reversible.

Everything runs locally against the in-store shopping assistant and its store tools (`agents/research_agent.py`). No server or Engine required for this walkthrough.

## Setup

In [ ]:
import sys, time
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / ".env", override=True)

from utils.models import model
from langsmith import uuid7
from langchain.agents import create_agent
from langchain_core.tools import tool

# Store tools shared with the rest of the workshop.
from agents.research_agent import (
    store_directory, check_stock, price_lookup, find_substitution, web_search,
)
print("Ready — tools:", [t.name for t in (store_directory, check_stock, price_lookup, find_substitution)])

---
# Part 1. Subagent Handoffs — the three patterns

The "sweet spot" question is really *how much context crosses the boundary, and how often you pay for a fresh model call*. Three patterns trade those off differently:

| Pattern | Context sharing | Cost | Use when |
|---|---|---|---|
| **Subagents as tools** (supervisor) | fresh context per call; subagent returns only its result | repeated model calls, more total tokens/request | independent sub-tasks; you want the main context clean; can fan out in parallel |
| **Handoffs via middleware state** | one agent, a state variable steers routing/config across turns | sequential → latency | you must enforce an order (can't refund before the warranty ID) |
| **Subgraphs with `Command.PARENT`** | you own exactly what history crosses the boundary | most engineering | a step genuinely needs its own complex graph |

The docs recommend **middleware handoffs over subgraphs** for most cases. Subagents-as-tools is the **anti-bloat** answer — the main agent never sees the subagent's intermediate tool chatter, only its final result.

### 1.1 Subagents as tools (supervisor pattern)

Each subagent is its own compiled agent with a **fresh context**; we wrap each as a `@tool` the supervisor calls. The supervisor keeps the conversation history; the subagent returns only its final string. Because they're just tools, the model can call **several in one turn in parallel** when the sub-tasks are independent.

In [ ]:
# Two focused subagents, each a full agent in its own context.
finder = create_agent(
    model=model, tools=[store_directory, check_stock],
    system_prompt="You locate one grocery item: aisle and stock. Be terse.",
)
pricer = create_agent(
    model=model, tools=[price_lookup, find_substitution],
    system_prompt="You price one grocery item and suggest a sub if it's out. Be terse.",
)

@tool
def find_item(item: str) -> str:
    """Delegate to the item-finder subagent (fresh context; returns only its result)."""
    r = finder.invoke({"messages": [{"role": "user", "content": f"Find {item}"}]})
    return r["messages"][-1].text

@tool
def price_item(item: str) -> str:
    """Delegate to the pricing subagent (fresh context; returns only its result)."""
    r = pricer.invoke({"messages": [{"role": "user", "content": f"Price {item}"}]})
    return r["messages"][-1].text

supervisor = create_agent(
    model=model,
    tools=[find_item, price_item],
    system_prompt=(
        "You are a shopping supervisor. Delegate to the find_item and price_item "
        "subagents. When sub-tasks are independent, call them in parallel in one turn. "
        "The subagents' internal steps never enter your context — you only see their results."
    ),
)

result = supervisor.invoke({"messages": [{"role": "user", "content": (
    "For salsa and tortillas: where are they, are they in stock, and what do they cost?"
)}]})
print(result["messages"][-1].text)

**Why this is the anti-bloat answer:** the supervisor's context contains the *request*, its *delegation tool calls*, and each subagent's *final result* — never the subagents' internal `check_stock` / `price_lookup` chatter. That chatter stays in each subagent's own (discarded) context. The cost is that each delegation is a fresh model loop, so you pay more total tokens per request. You'll see exactly that split in the trace in §1.4.

### 1.2 Handoffs via middleware state (one agent, steered by a state variable)

When you must **enforce an order** — the classic *"can't process a refund before the warranty ID is collected"* — a supervisor's parallel, stateless calls are the wrong tool. Instead, keep **one agent** and carry a **state variable** that gates what's allowed. Middleware reads/writes that state across turns. Fewer repeat model calls than the supervisor; the cost is that it's sequential.

Below: a `warranty_id` state field. `process_refund` is **gated** by middleware until `collect_warranty` has populated it — the order is enforced in code, not left to the model's goodwill.

In [ ]:
from langchain.agents.middleware import AgentMiddleware, AgentState
from langchain_core.tools import InjectedToolCallId
from langgraph.types import Command
from typing_extensions import Annotated, NotRequired

class ReturnsState(AgentState):
    warranty_id: NotRequired[str]   # the state variable that steers routing

@tool
def collect_warranty(warranty_id: str, tool_call_id: Annotated[str, InjectedToolCallId]) -> Command:
    """Record the warranty ID. Must happen before any refund."""
    return Command(update={
        "warranty_id": warranty_id,
        "messages": [{"role": "tool", "content": f"Warranty {warranty_id} on file.",
                      "tool_call_id": tool_call_id}],
    })

@tool
def process_refund(item: str) -> str:
    """Refund an item. Requires a warranty ID already on file."""
    return f"Refund approved for {item}."

class RefundOrderGate(AgentMiddleware):
    """Hard sequential constraint: no refund until a warranty ID is in state."""
    state_schema = ReturnsState

    def wrap_tool_call(self, request, handler):
        if request.tool_call["name"] == "process_refund" and not request.state.get("warranty_id"):
            # Short-circuit: return a tool message instead of running the tool.
            return {"role": "tool",
                    "content": "BLOCKED: collect the warranty ID first (call collect_warranty).",
                    "tool_call_id": request.tool_call["id"]}
        return handler(request)

returns_agent = create_agent(
    model=model,
    tools=[collect_warranty, process_refund],
    system_prompt="You handle grocery returns. Collect the warranty ID, then process the refund.",
    middleware=[RefundOrderGate()],
)

out = returns_agent.invoke({"messages": [{"role": "user", "content": (
    "Refund my rotisserie chicken, warranty ID WZ-4471."
)}]})
print(out["messages"][-1].text)
print("\nwarranty_id in final state:", out.get("warranty_id"))

The constraint holds **regardless of what the model tries** — if it calls `process_refund` first, the middleware short-circuits with a BLOCKED tool message and the model must go collect the warranty ID. That's the difference between "we asked the prompt nicely" and "the order is guaranteed."

### 1.3 Subgraphs with `Command.PARENT` (own the boundary)

Reach for this **only when a step genuinely needs its own complex graph**. Then *you* own the context engineering: what history crosses the boundary, and pairing each `AIMessage` (tool call) with its `ToolMessage` so the receiving agent has a **valid** message history. Get that pairing wrong and the receiving model sees a dangling tool call.

A node returns `Command(goto=..., graph=Command.PARENT, update=...)` to route in the *parent* graph and pass a curated slice of state up. Below is the shape (a fixed sub-step that hands one summarized line back to the parent), kept deterministic so it runs without a model.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing_extensions import TypedDict, Literal

class TripState(TypedDict):
    request: str
    handoff_note: str   # the ONLY thing we let cross the boundary

# A subgraph node that does its own work, then hands a curated one-liner to the
# parent via Command.PARENT — not its full internal state.
def budget_subgraph(state: TripState) -> Command[Literal["__end__"]]:
    items = [w.strip() for w in state["request"].split(",") if w.strip()]
    note = f"Budget check: {len(items)} items on the list; none over the $10 flag."
    # graph=Command.PARENT routes in the parent graph; update carries only `note`.
    return Command(goto=END, graph=Command.PARENT, update={"handoff_note": note})

sub = StateGraph(TripState)
sub.add_node("budget", budget_subgraph)
sub.add_edge(START, "budget")
budget_graph = sub.compile()

parent = StateGraph(TripState)
parent.add_node("budget_step", budget_graph)   # embed the subgraph as a node
parent.add_edge(START, "budget_step")
parent.add_edge("budget_step", END)
trip = parent.compile()

print(trip.invoke({"request": "milk, eggs, salsa, tortillas", "handoff_note": ""})["handoff_note"])
print("\nOnly `handoff_note` crossed the boundary — you chose that, not the framework.")

### 1.4 Read it in the trace — is the subagent earning its tokens?

The tradeoff table is abstract until you see it per span. Run one supervisor request, then open the trace in LangSmith and walk the **nesting**: each subagent is a nested run with its own **tokens, latency, and cost**, and its own **inputs** (exactly the context it received).

This is the evidence that answers *"should we even have made this a subagent?"*:
- A subagent whose token/latency cost dwarfs the value of its one-line result **isn't earning its tokens** — fold it back into the parent.
- A hop where the subagent's **input is missing context** the parent had — that's a **losing-context** handoff; widen what you pass or switch to a middleware handoff that keeps shared state.

> If your team's POC traces are already in this workspace, do this walk on **their** data, not the demo — the per-span cost/context story lands harder on real traffic.

In [ ]:
# Run a supervisor request with tracing on; then open the trace and inspect the
# nested subagent spans. (Tracing is automatic when LANGSMITH_TRACING=true.)
import os
traced = supervisor.invoke({"messages": [{"role": "user", "content": (
    "Compare salsa and tortillas: aisle, stock, and price for each."
)}]})
print(traced["messages"][-1].text[:400])

proj = os.environ.get("LANGSMITH_PROJECT", "default")
print(f"\nOpen LangSmith project '{proj}' and expand this run.")
print("For each nested subagent span, read: tokens, latency, cost, and its Inputs")
print("(the exact context it received). That is the earning-its-tokens evidence.")

**What to look for in the nesting:**

```
supervisor (root)                 <- full history: request + delegations + results
├── find_item  -> finder          <- fresh context: just "Find salsa"
│   ├── check_stock                    (subagent's internal chatter — NOT in supervisor)
│   └── store_directory
├── price_item -> pricer          <- fresh context: just "Price salsa"
│   └── price_lookup
└── ...                           <- parallel calls appear as siblings
```

Group the project by run name to compare total tokens across subagents; if `pricer` costs as much as `finder` but only returns a price string, that's your "not earning its tokens" signal.

### 1.5 Background subagent — don't block the chat turn

Sometimes a subagent's work is slow (a full week's meal-plan + route) and the shopper shouldn't wait. On a **deployed** server you spawn the subagent's work as a **background run** so the chat turn returns immediately and the result lands later. This is the same durable-execution mechanism from the Server Features module — it just doubles as the "non-blocking subagent" answer.

The shape (against a `langgraph dev` / deployment, as in the Server Features module):

```python
from langgraph_sdk import get_sync_client
client = get_sync_client(url="http://localhost:2024")

# Chat turn: answer now.
quick = client.runs.wait(None, assistant_id,
    input={"messages": [{"role": "user", "content": "Start planning my week; I'll check back."}]})

# Background subagent work: returns a run id immediately, keeps working server-side.
bg = client.runs.create(thread_id, assistant_id,
    input={"messages": [{"role": "user", "content": "Build the full week meal-plan + aisle route."}]})
# ... later ...
plan = client.runs.join(thread_id, bg["run_id"])   # collect when ready
```

Locally (no server), the analog is simply invoking the subagent on its own thread and joining later — but the durable, survives-disconnect version is a server run. See **Module 6 §1 (Background jobs)** and **§3 (Durable execution)**.

---
# Part 2. Deterministic Behavior — the dial

*"Can I make it more deterministic for specific pathways, so a repeat query does fewer tool calls and less re-deciding?"* Yes — but treat determinism as a **dial** you turn per pathway, not a global switch. Go down it in order; each step removes more non-determinism than the last, at the cost of flexibility.

1. **Put determinism in code** where it matters (custom graph: fixed steps aren't model decisions at all).
2. **Constrain the model** where it stays agentic (state order, structured output, tool gating, HITL as a hard stop).
3. **Self-improving loop** (Engine): cluster recurring failures, write the skill/context that removes repeated tool calls. *Honest caveat below.*
4. **Version the behavior** so "we made it more deterministic" is promotable and reversible.

### 2.1 Put determinism in code where it matters

The strongest answer for anyone worried about a headline-making mistake: **the parts that must be identical every time shouldn't be a model decision at all.** A custom `StateGraph` mixes **fixed** nodes (routing, validation, an external API call) with **agentic** nodes. The fixed nodes run the same way every time — zero model variance.

Below: a shopping-trip graph where parsing and a safety/validation gate are **fixed code**, and only the final "explain the route" step is agentic. The validation node can hard-stop before any agentic step ever runs.

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict, Literal

class RouteState(TypedDict):
    request: str
    items: list
    rejected: str
    answer: str

# Items we will never auto-substitute or reroute without a human (safety pathway).
RESTRICTED = {"bleach", "ammonia"}   # e.g. never suggest mixing/adjacent placement

def parse(state: RouteState):                      # FIXED
    items = [w.strip().lower() for w in state["request"].split(",") if w.strip()]
    return {"items": items}

def safety_gate(state: RouteState) -> Command:     # FIXED: deterministic hard rule
    bad = sorted(set(state["items"]) & RESTRICTED)
    if bad:
        return Command(goto=END, update={"rejected": f"Refusing to auto-handle restricted items: {bad}"})
    return Command(goto="plan")

def plan(state: RouteState):                       # AGENTIC (only node that would call the model)
    return {"answer": "Aisle-ordered route for: " + ", ".join(state["items"])}

g = StateGraph(RouteState)
g.add_node("parse", parse); g.add_node("safety_gate", safety_gate); g.add_node("plan", plan)
g.add_edge(START, "parse"); g.add_edge("parse", "safety_gate"); g.add_edge("plan", END)
route_graph = g.compile()

print(route_graph.invoke({"request": "milk, eggs, salsa"})["answer"])
print(route_graph.invoke({"request": "milk, bleach, ammonia"})["rejected"])
print("\nThe safety gate is code — it decides identically every run, no model involved.")

### 2.2 Constrain the model where it stays agentic

Where you keep an agent, narrow its freedom so repeats behave the same:

- **State-driven order** — the middleware gate from §1.2 (can't skip a required step).
- **Structured output** — force the shape of the answer (`with_structured_output` / response format), so downstream code parses it deterministically.
- **Tool gating / limits** — cap tool calls so a pathway can't wander (`ToolCallLimitMiddleware`), or hide tools that don't apply.
- **Human-in-the-loop as a hard stop** — the run pauses on an interrupt and waits for a person in the **Agent Inbox** before continuing.

Below: a tool-call limit (fewer, more predictable calls) plus HITL on the refund so a human approves the irreversible step.

In [ ]:
from langchain.agents.middleware import ToolCallLimitMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import MemorySaver

constrained = create_agent(
    model=model,
    tools=[collect_warranty, process_refund],
    system_prompt="You handle grocery returns. Collect the warranty ID, then refund.",
    middleware=[
        RefundOrderGate(),                                   # order (from 1.2)
        ToolCallLimitMiddleware(thread_limit=6),             # bounded, more predictable
        HumanInTheLoopMiddleware(interrupt_on={"process_refund": True}),  # hard stop
    ],
    checkpointer=MemorySaver(),   # HITL interrupts require a checkpointer
)

cfg = {"configurable": {"thread_id": str(uuid7())}}
res = constrained.invoke({"messages": [{"role": "user", "content": (
    "Refund my birthday cake, warranty ID WZ-9001."
)}]}, config=cfg)

if res.get("__interrupt__"):
    print("PAUSED for human approval (this would appear in the Agent Inbox):")
    for it in res["__interrupt__"]:
        val = it.value if hasattr(it, "value") else it
        print("  awaiting approval on:", val)
    print("\nA human approves/edits/rejects in the Agent Inbox; the run resumes from the checkpoint.")
else:
    print(res["messages"][-1].text)

The interrupt is the **hard stop**: the irreversible action can't happen until a human acts. In a deployed setup this lands in the **Agent Inbox** (Studio / LangSmith), and you resume with `Command(resume=...)` exactly as in Module 6 §5.

### 2.3 The self-improving loop (Engine) — with the honest caveat

The dial's next notch is what was actually asked for: an agent that gets **more deterministic on repeated pathways over time**. That's **LangSmith Engine** (Module 5) working over your traces:

1. Cluster a recurring failure/inefficiency (e.g. the same three redundant `store_directory` calls on every taco request).
2. Write the fix as a **skill or context change** — a `SKILL.md` or an `AGENTS.md` line that tells a subagent "for taco requests, these are the four items and aisles" — so the next similar query does **fewer tool calls and less re-deciding**.
3. Ship it as a **diff for review**, then deploy an **evaluator** so a regression **reopens** the issue.

**Be honest about the boundary:** Engine changes the agent's **context and prompts** (skills, memory, system guidance) — it does **not** rewrite tool-call *ordering in code*. So it makes pathways more consistent by giving the model better standing context, not by hard-coding a tool sequence. If you need guaranteed ordering, that's §2.1 (put it in code) or §2.2 (state gate) — not Engine.

*We're not enabling Engine in this module — see **Module 5** to run the detect → diagnose → fix → guard loop live.*

### 2.4 Version the behavior — promotable, reversible

Once you've tuned a pathway (a skill, a context file, a prompt), *"we made it more deterministic"* should be a **governed, reversible change** — not a prompt someone edited on a Friday. Two mechanisms, both from earlier modules:

- **Context Hub** holds the behavior file (the `AGENTS.md` / `SKILL.md`) with **commits** and **dev / staging / prod** tags. A change is a commit you can promote or roll back — and it can fire a **Context Hub commit webhook** (Module 6 §7.4) into the platform team's review pipeline.
- **Assistants version the config** (Module 6 §6): the prompt/config that encodes the behavior is pinned per assistant, promoted with `set_latest`, and rolled back the same way.

This is the same governance thread as the A/B story — so "make it more deterministic" ships like any other reviewed, versioned, roll-back-able change.

---
## Recap

**Part 1 — Subagent handoffs.** Three patterns, chosen by the context/cost tradeoff:

| Pattern | Anti-bloat? | Enforces order? | Cost |
|---|---|---|---|
| Subagents as tools (supervisor) | ✅ fresh context, result-only | no (parallel/stateless) | more model calls / tokens |
| Middleware-state handoff | partial (shared state) | ✅ sequential gate | latency (sequential) |
| Subgraph + `Command.PARENT` | you decide | ✅ if you build it | most engineering |

Then **read the trace**: per-subagent tokens/latency/cost and the exact context each received tell you whether a subagent earns its tokens or a hop loses context. Slow subagent work goes **background** so the chat turn returns (Module 6 §1/§3).

**Part 2 — Deterministic behavior is a dial:** put it in **code** (§2.1) → **constrain** the model (§2.2) → **self-improving loop** via Engine on context/prompts, *not* code-level tool ordering (§2.3) → **version** the behavior in Context Hub + assistants (§2.4).

**Docs:** [Subagents](https://docs.langchain.com/oss/python/langchain/subagents) · [Handoffs / middleware](https://docs.langchain.com/oss/python/langchain/middleware) · [Subgraphs](https://docs.langchain.com/oss/python/langgraph/subgraphs) · [HITL](https://docs.langchain.com/oss/python/langchain/human-in-the-loop) · [Engine](https://docs.langchain.com/langsmith/engine)